In [16]:
from pathlib import Path
import shutil
import re
import unicodedata
import pandas as pd

RAW_DIR = Path("../raw")
INTERIM_DIR = Path("../interim")

CONSTITUENCY_DIR = RAW_DIR / "lok_sabha_constituencies"
CANDIDATES_DIR = RAW_DIR / "lok_sabha_candidate_dfs"

YEARS = [2014, 2019, 2024]

CONSTITUENCY_COLUMNS = [
    "state_ut", "state_id", "const",
    "const_id", "link", "election_date"
]

BYE_PATTERN = re.compile(
    r"\bBYE[\s\-\u2010-\u2015]*ELECTION\b",
    flags=re.IGNORECASE
)

In [17]:
assert RAW_DIR.exists(), f"Raw directory not found: {RAW_DIR}"
assert CONSTITUENCY_DIR.exists(), f"Missing: {CONSTITUENCY_DIR}"
assert CANDIDATES_DIR.exists(), f"Missing: {CANDIDATES_DIR}"

for year in YEARS:
    path = CONSTITUENCY_DIR / f"constituency_links_{year}.csv"
    assert path.exists(), f"Missing constituency CSV: {path}"

candidate_files = sorted(CANDIDATES_DIR.rglob("candidates.csv"))
assert candidate_files, f"No candidates.csv files found under {CANDIDATES_DIR}"

print(f"Candidate CSV files: {len(candidate_files)}")
print(f"Raw directory: {RAW_DIR.resolve()}")

Candidate CSV files: 12
Raw directory: D:\MyCoding\DA2402\DCV-Project\raw


In [18]:
constituency_dfs = {}

for year in YEARS:
    path = CONSTITUENCY_DIR / f"constituency_links_{year}.csv"

    df = pd.read_csv(
        path,
        dtype={"state_id": str, "const_id": str}
    )

    missing = set(CONSTITUENCY_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"{year}: missing columns {missing}")

    df = df[CONSTITUENCY_COLUMNS].copy()

    normalized_const = (
        df["const"].fillna("")
        .str.replace("\u00a0", " ", regex=False)
    )

    df["bye_election"] = normalized_const.str.contains(
        BYE_PATTERN,
        na=False
    )

    assert df["const_id"].notna().all()
    assert df["link"].notna().all()

    constituency_dfs[year] = df

    print(
        f"{year}: {len(df)} total, "
        f"{(~df['bye_election']).sum()} general, "
        f"{df['bye_election'].sum()} bye-election"
    )

2014: 574 total, 543 general, 31 bye-election
2019: 559 total, 543 general, 16 bye-election
2024: 546 total, 543 general, 3 bye-election


In [19]:
for path in candidate_files:
    relative_path = path.relative_to(CANDIDATES_DIR)
    df = pd.read_csv(path, dtype=str)

    print(f"\n{relative_path}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2).to_string(index=False))


2014\BIHAR\candidates.csv
Rows: 41
Columns: ['year', 'state', 'state_id', 'const', 'const_id', 'candidate', 'candidate_id', 'candidate_url', 'is_winner', 'party', 'criminal_cases', 'education', 'age', 'total_assets', 'asset_link', 'liabilities', 'constituency_url', 'html_path']
year state state_id                               const const_id     candidate candidate_id                                                   candidate_url is_winner party criminal_cases education age              total_assets                                                                     asset_link             liabilities                                                                    constituency_url                                                                   html_path
2014 BIHAR        4                              ARARIA      304  Taslim Uddin         4456  https://www.myneta.info/ls2014/candidate.php?candidate_id=4456      True   RJD              4  Literate  72                       NaN htt

In [21]:
if INTERIM_DIR.exists():
    raise FileExistsError(
        f"{INTERIM_DIR} already exists. "
        "Inspect it before deciding whether to replace it."
    )

INTERIM_DIR.mkdir(parents=True)

copied_files = 0
skipped_html = 0

for source in RAW_DIR.rglob("*"):
    relative_path = source.relative_to(RAW_DIR)
    destination = INTERIM_DIR / relative_path

    if source.is_dir():
        destination.mkdir(parents=True, exist_ok=True)
        continue

    if source.suffix.lower() in {".html", ".htm"}:
        skipped_html += 1
        continue

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    copied_files += 1

print(f"Created: {INTERIM_DIR.resolve()}")
print(f"Copied non-HTML files: {copied_files}")
print(f"HTML files excluded: {skipped_html}")

Created: D:\MyCoding\DA2402\DCV-Project\interim
Copied non-HTML files: 18
HTML files excluded: 640


In [22]:
for year, df in constituency_dfs.items():
    output_path = (
        INTERIM_DIR
        / "lok_sabha_constituencies"
        / f"constituency_links_{year}.csv"
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    print(f"Saved {output_path}")

Saved ..\interim\lok_sabha_constituencies\constituency_links_2014.csv
Saved ..\interim\lok_sabha_constituencies\constituency_links_2019.csv
Saved ..\interim\lok_sabha_constituencies\constituency_links_2024.csv


In [23]:
for year in YEARS:
    path = (
        INTERIM_DIR
        / "lok_sabha_constituencies"
        / f"constituency_links_{year}.csv"
    )

    df = pd.read_csv(path)

    expected_bye_counts = {
        2014: 31,
        2019: 16,
        2024: 3,
    }

    assert len(df) == 543 + expected_bye_counts[year]
    assert df["bye_election"].sum() == expected_bye_counts[year]
    assert df["bye_election"].notna().all()

    print(
        f"{year}: {len(df)} rows | "
        f"{(~df['bye_election']).sum()} general | "
        f"{df['bye_election'].sum()} bye-election"
    )

2014: 574 rows | 543 general | 31 bye-election
2019: 559 rows | 543 general | 16 bye-election
2024: 546 rows | 543 general | 3 bye-election


In [24]:
for path in sorted(INTERIM_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(INTERIM_DIR))

lok_sabha_candidate_dfs\2014\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2014\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2014\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2014\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\2019\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2019\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2019\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2019\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\2024\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2024\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2024\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2024\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\candidates_all_years.csv
lok_sabha_constituencies\constituency_links_2014.csv
lok_sabha_constituencies\constituency_links_2019.csv
lok_sabha_constituencies\constituency_links_2024.csv
lok_sabha_constituencies\constituency_links_all_years.csv
lok_sabha_html\html_manifest.csv


In [26]:
constituency_lookup = {}

for year, df in constituency_dfs.items():
    lookup = df.copy()

    lookup["const_id"] = lookup["const_id"].astype(str).str.strip()
    lookup["state_id"] = lookup["state_id"].astype(str).str.strip()

    if lookup["const_id"].duplicated().any():
        duplicates = lookup.loc[
            lookup["const_id"].duplicated(keep=False),
            ["state_ut", "const", "const_id", "link"]
        ]
        raise ValueError(
            f"Duplicate constituency IDs for {year}:\n{duplicates}"
        )

    constituency_lookup[year] = lookup.set_index("const_id")

    print(
        f"{year}: {len(lookup)} constituency records, "
        f"{lookup['bye_election'].sum()} by-elections"
    )

2014: 574 constituency records, 31 by-elections
2019: 559 constituency records, 16 by-elections
2024: 546 constituency records, 3 by-elections


In [27]:
candidate_summary = []

for source_path in sorted(CANDIDATES_DIR.rglob("candidates.csv")):
    relative_path = source_path.relative_to(CANDIDATES_DIR)
    year = int(relative_path.parts[0])

    candidates = pd.read_csv(source_path, dtype=str)
    candidates["const_id"] = candidates["const_id"].astype(str).str.strip()
    candidates["state_id"] = candidates["state_id"].astype(str).str.strip()

    lookup = constituency_lookup[year]

    missing_ids = set(candidates["const_id"]) - set(lookup.index)

    if missing_ids:
        raise ValueError(
            f"{year} {relative_path}: constituency IDs not found: "
            f"{sorted(missing_ids)[:10]}"
        )

    mapped = candidates["const_id"].map(lookup["bye_election"])

    if mapped.isna().any():
        raise ValueError(
            f"Unmatched constituency flags in {relative_path}"
        )

    mapped_state = candidates["const_id"].map(lookup["state_id"])

    state_mismatch = candidates["state_id"] != mapped_state

    if state_mismatch.any():
        examples = candidates.loc[
            state_mismatch,
            ["state", "state_id", "const", "const_id"]
        ].head(10)

        raise ValueError(
            f"State mismatch in {relative_path}:\n{examples}"
        )

    # If a candidate CSV contains a constituency URL, verify the join.
    if "constituency_url" in candidates.columns:
        mapped_urls = candidates["const_id"].map(lookup["link"])

        candidate_urls = candidates["constituency_url"].fillna("").str.strip()
        expected_urls = mapped_urls.fillna("").str.strip()

        url_mismatch = (
            candidate_urls.ne("")
            & expected_urls.ne("")
            & candidate_urls.ne(expected_urls)
        )

        if url_mismatch.any():
            examples = candidates.loc[
                url_mismatch,
                ["const", "const_id", "constituency_url"]
            ].head(10)

            raise ValueError(
                f"Constituency URL mismatch in {relative_path}:\n{examples}"
            )

    candidates["bye_election"] = mapped.astype(bool)

    output_path = INTERIM_DIR / "lok_sabha_candidate_dfs" / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    candidates.to_csv(output_path, index=False)

    candidate_summary.append({
        "year": year,
        "file": str(relative_path),
        "candidates": len(candidates),
        "bye_election_candidates": int(candidates["bye_election"].sum()),
        "general_candidates": int((~candidates["bye_election"]).sum()),
    })

    print(
        f"Saved {output_path} | "
        f"candidates={len(candidates)} | "
        f"bye={candidates['bye_election'].sum()} | "
        f"general={(~candidates['bye_election']).sum()}"
    )

Saved ..\interim\lok_sabha_candidate_dfs\2014\BIHAR\candidates.csv | candidates=41 | bye=1 | general=40
Saved ..\interim\lok_sabha_candidate_dfs\2014\MAHARASHTRA\candidates.csv | candidates=50 | bye=3 | general=47
Saved ..\interim\lok_sabha_candidate_dfs\2014\TAMIL_NADU\candidates.csv | candidates=39 | bye=0 | general=39
Saved ..\interim\lok_sabha_candidate_dfs\2014\UTTAR_PRADESH\candidates.csv | candidates=83 | bye=4 | general=79
Saved ..\interim\lok_sabha_candidate_dfs\2019\BIHAR\candidates.csv | candidates=42 | bye=2 | general=40
Saved ..\interim\lok_sabha_candidate_dfs\2019\MAHARASHTRA\candidates.csv | candidates=49 | bye=1 | general=48
Saved ..\interim\lok_sabha_candidate_dfs\2019\TAMIL_NADU\candidates.csv | candidates=40 | bye=1 | general=39
Saved ..\interim\lok_sabha_candidate_dfs\2019\UTTAR_PRADESH\candidates.csv | candidates=82 | bye=3 | general=79
Saved ..\interim\lok_sabha_candidate_dfs\2024\BIHAR\candidates.csv | candidates=144 | bye=0 | general=144
Saved ..\interim\lok_sab

In [28]:
candidate_summary_df = pd.DataFrame(candidate_summary)

display(candidate_summary_df)

for source_path in sorted(CANDIDATES_DIR.rglob("candidates.csv")):
    relative_path = source_path.relative_to(CANDIDATES_DIR)
    output_path = INTERIM_DIR / "lok_sabha_candidate_dfs" / relative_path

    original = pd.read_csv(source_path, dtype=str)
    processed = pd.read_csv(output_path, dtype=str)

    assert len(original) == len(processed), relative_path
    assert "bye_election" in processed.columns
    assert processed["bye_election"].notna().all()

    # Every original column must still be present.
    assert set(original.columns).issubset(processed.columns)

print("All candidate CSVs passed basic validation.")

,year,file,candidates,bye_election_candidates,general_candidates
0,2014,2014\BIHAR\candidates.csv,41,1,40
1,2014,2014\MAHARASHTRA\candidates.csv,50,3,47
2,2014,2014\TAMIL_NADU\candidates.csv,39,0,39
3,2014,2014\UTTAR_PRADESH\candidates.csv,83,4,79
4,2019,2019\BIHAR\candidates.csv,42,2,40
5,2019,2019\MAHARASHTRA\candidates.csv,49,1,48
6,2019,2019\TAMIL_NADU\candidates.csv,40,1,39
7,2019,2019\UTTAR_PRADESH\candidates.csv,82,3,79
8,2024,2024\BIHAR\candidates.csv,144,0,144
9,2024,2024\MAHARASHTRA\candidates.csv,49,1,48


All candidate CSVs passed basic validation.


In [29]:
for path in sorted(INTERIM_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(INTERIM_DIR))

html_files = [
    path for path in INTERIM_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".html", ".htm"}
]

assert not html_files, f"HTML files unexpectedly copied: {html_files[:5]}"

print("\nInterim directory verified: no HTML response files.")

lok_sabha_candidate_dfs\2014\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2014\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2014\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2014\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\2019\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2019\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2019\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2019\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\2024\BIHAR\candidates.csv
lok_sabha_candidate_dfs\2024\MAHARASHTRA\candidates.csv
lok_sabha_candidate_dfs\2024\TAMIL_NADU\candidates.csv
lok_sabha_candidate_dfs\2024\UTTAR_PRADESH\candidates.csv
lok_sabha_candidate_dfs\candidates_all_years.csv
lok_sabha_constituencies\constituency_links_2014.csv
lok_sabha_constituencies\constituency_links_2019.csv
lok_sabha_constituencies\constituency_links_2024.csv
lok_sabha_constituencies\constituency_links_all_years.csv
lok_sabha_html\html_manifest.csv

Interim directory verified: no HTML respons